In [2]:
from getpass import getpass
from dotenv import load_dotenv
import os

load_dotenv("/Users/patrick/Documents/00-AiSpace /AI Agent/.env", override=True)


ACCESS_TOKEN = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

In [3]:
from langchain_community.document_loaders import GitHubIssuesLoader

loader = GitHubIssuesLoader(repo="huggingface/peft", access_token=ACCESS_TOKEN, include_prs=False, state="all")

docs = loader.load()

/var/folders/pj/6hcpl0bs1h59nsc78hkrgz980000gn/T/ipykernel_26172/1730161545.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import GitHubIssuesLoader


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=30)

chunked_docs = splitter.split_documents(docs)

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(chunked_docs, HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5"))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v0.3"
#qwen2:7b

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

W1121 16:32:50.548000 26172 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/69.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

In [7]:
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import pipeline
from langchain_core.output_parsers import StrOutputParser

text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=True,
    max_new_tokens=400,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<|system|>
Answer the question based on your knowledge. Use the following context to help:

{context}

</s>
<|user|>
{question}
</s>
<|assistant|>

 """

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)

llm_chain = prompt | llm | StrOutputParser()


In [8]:
from langchain_core.runnables import RunnablePassthrough

retriever = db.as_retriever()

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | llm_chain

In [ ]:
question = "How do you combine multiple adapters?"

response = llm_chain.invoke({"context": "", "question": question})
print("Response:", response)


Response: 
<|system|>
Answer the question based on your knowledge. Use the following context to help:



</s>
<|user|>
How do you combine multiple adapters?
</s>
<|assistant|>

 1. Connect the adapter to the power source.
2. Plug the adapter into a USB port on your computer or device.
3. Follow the manufacturer's instructions for connecting and configuring the adapter.
4. You can use any of the available USB adapters to connect to different devices, such as laptops, phones, tablets, cameras, and more.
5. To disconnect an adapter from its USB port, follow the steps outlined in step 2. Make sure the adapter is properly connected before plugging it back in.
6. If you have any other questions or need further assistance, feel free to ask.


































































































































































































































































: 

In [ ]:
response = rag_chain.invoke(question)
print("Response:", response)

In [ ]:
#Apple Optimized for M1 chip

In [ ]:

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

# Model configuration
model_name = "Qwen/Qwen3-0.6B"

# Device configuration for Apple Silicon
device = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Using device: {device}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "mps" else torch.float32,
)

model = model.to(device)
model.eval()

# Example inference
prompt = "Explain how PostgreSQL indexing works."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(response)
